# BRCA 1 Concordance Script

### This script evaluates concordance between PolyPhen-2 and REVEL predictions and also compares each model’s predictions to the variant’s clinical significance reported in ClinVar

There is also concordance check between student predictions. We had each student predict with Polyphen-2 to add redunancy into our validation and make sure everyone was getting the same results.

In [1]:
#importing the packages needed
import pandas as pd

In [ ]:
#reading in the completed file
#this file is the Copy-missense filter(Polyphen) tab on the Copy of 2026_BRCA1_variants_amr_parsed_PP2_CT_ file (the BRCA1 master spreadsheet all students used)
#the tab was copy and pasted into another spreadsheet to upload. This is because if we want to run in AOU, it only excepts CSVs, which can only have one tab

#Reading in the data
#For AOU you may need to use a CSV file or run a Pip install of Openpyxl
BRCA1_UG_AN = pd.read_excel("C:/Users/knigh/Documents/UHD MSDA/Capstone/Concordance/2026_BRCA1_UgCompleted.xlsx")
BRCA1_UG_AN.head()
#this file is pretty big dimension wise. The file I am using has 52 columns. It might we helpful to run this script in Spyder or VScode to see the data in full if needed

#Data Preview removed to comply with All of Us data user policies. You can see all the data's feature in the code block below.

In [3]:
#since the file has 52 columns, lets only save the columns we are interested in
#listing the column names
BRCA1_UG_AN.columns

Index(['locus', 'alleles', 'qc_AC', 'qc_AF', 'qc_AN', 'call_rate', 'n_het',
       'info_AC', 'info_AF', 'info_AN', 'vid', 'contig', 'position',
       'consequence', 'clinvar_classification', 'variant_type', 'ref_allele',
       'alt_allele', 'gvs_all_ac', 'gvs_all_an', 'gvs_all_af', 'gvs_all_sc',
       'dbsnp_rsid', 'revel', 'vid_alt', 'gvs_max_subpop', 'aa_change',
       'transcript', 'gene_symbol', 'gvs_max_ac', 'gvs_max_sc', 'gvs_max_an',
       'gnomad_max_subpop', 'protein_id', 'hgvsp', 'aa_extract', 'AA1', 'AA2',
       'Protein', 'Position', 'Ref_AA', 'Alt_AA',
       '(HumDiv) Polyphen Significance(S1)', 'Score(S1)', 'Sensitivity(S1)',
       'FPR Score(S1)', 'Specificity(S1)',
       '(HumDiv) Polyphen Significance(S2)', 'Score(S2)', 'Sensitivity(S2)',
       'FPR Score(S2)', 'Specificity(S2)'],
      dtype='object')

In [ ]:
#creating a subset of only the desired columns
BRCA1_SS = BRCA1_UG_AN[["vid","consequence","clinvar_classification","variant_type","dbsnp_rsid","revel",'(HumDiv) Polyphen Significance(S1)', 'Score(S1)', 'Sensitivity(S1)',
       'FPR Score(S1)', 'Specificity(S1)',
       '(HumDiv) Polyphen Significance(S2)', 'Score(S2)', 'Sensitivity(S2)',
       'FPR Score(S2)', 'Specificity(S2)']]

#brought down to 16 columns
BRCA1_SS.head() #again probably better to view in Syder or VScode if able to see full set

#There are no AOU participant identitifers in the preview below

,vid,consequence,clinvar_classification,variant_type,dbsnp_rsid,revel,(HumDiv) Polyphen Significance(S1),Score(S1),Sensitivity(S1),FPR Score(S1),Specificity(S1),(HumDiv) Polyphen Significance(S2),Score(S2),Sensitivity(S2),FPR Score(S2),Specificity(S2)
0,17-43049131-G-T,missense_variant,not provided,SNV,rs786201945,0.562,Possibly Damaging,0.882,0.0644,0.824,0.176,Possibly Damaging,0.882,0.0644,0.824,0.176
1,17-43049153-C-T,missense_variant,uncertain significance,SNV,rs1555575131,0.690,Benign,0.063,0.1600,0.937,0.063,Benign,0.063,0.1600,0.937,0.063
2,17-43049164-C-T,missense_variant,"uncertain significance, likely pathogenic, pat...",SNV,rs80357069,0.862,Probably Damaging,0.997,0.0167,0.409,0.591,Probably Damaging,0.997,0.0167,0.409,0.591
3,17-43049167-C-T,missense_variant,"uncertain significance, likely pathogenic, not...",SNV,rs1597801649,0.789,Possibly Damaging,0.917,0.0597,0.812,0.188,Possibly Damaging,0.917,0.0597,0.812,0.188
4,17-43049168-A-T,missense_variant,"uncertain significance, pathogenic, not provided",SNV,rs80357065,0.812,Probably Damaging,0.986,0.0368,0.736,0.264,Probably Damaging,0.986,0.0368,0.736,0.264


In [5]:
#checking for NAs
BRCA1_SS.isna().sum()

#there are 28 NA's for clinvar_classfication and 12 for rsid. Not worries about rsid, but will probably need to look into clinvar_class. But no other NA's is great!
#spot checked the variants missing clinvar_class in the AOU data browser and Gnomad, they either had a blank significance or did not exist in gnomad

vid                                    0
consequence                            0
clinvar_classification                28
variant_type                           0
dbsnp_rsid                            12
revel                                  0
(HumDiv) Polyphen Significance(S1)     0
Score(S1)                              0
Sensitivity(S1)                        0
FPR Score(S1)                          0
Specificity(S1)                        0
(HumDiv) Polyphen Significance(S2)     0
Score(S2)                              0
Sensitivity(S2)                        0
FPR Score(S2)                          0
Specificity(S2)                        0
dtype: int64

## Aligning the columns values for comparisons

In [6]:
#checking the unique values in the consequence column. Should all be variations of missense
BRCA1_SS.consequence.unique()

array(['missense_variant', 'missense_variant, splice_region_variant'],
      dtype=object)

In [7]:
#just changing all the missense to be missense_variant
BRCA1_SS.loc[BRCA1_SS["consequence"].str.contains("missense"), "consequence"] = "missense_variant"

#checking it worked
print(BRCA1_SS.consequence.unique())

#grabbing the count of missense values, should be the same as the # of rows
(BRCA1_SS['consequence']=='missense_variant').sum() #426

['missense_variant']


np.int64(426)

In [8]:
#looking at the values in the clinvar classification columns
print(BRCA1_SS.clinvar_classification.unique())

#looking at the count of the values so when we condense them, we can make sure the count was allocated correctly
BRCA1_SS["clinvar_classification"].value_counts()

['not provided' 'uncertain significance'
 'uncertain significance, likely pathogenic, pathogenic'
 'uncertain significance, likely pathogenic, not provided'
 'uncertain significance, pathogenic, not provided'
 'benign, likely benign, uncertain significance'
 'likely benign, uncertain significance'
 'likely benign, uncertain significance, not provided'
 'uncertain significance, not provided' 'benign, likely benign'
 'likely pathogenic, pathogenic' 'pathogenic'
 'likely pathogenic, pathogenic, low penetrance'
 'benign, uncertain significance' 'benign' 'likely benign' nan
 'uncertain significance, likely pathogenic'
 'benign, likely benign, uncertain significance, likely pathogenic'
 'benign, uncertain significance, not provided']


clinvar_classification
likely benign, uncertain significance                               194
benign, likely benign, uncertain significance                        71
benign, likely benign                                                46
uncertain significance                                               39
likely benign, uncertain significance, not provided                  11
uncertain significance, not provided                                  8
benign                                                                5
pathogenic                                                            4
benign, uncertain significance                                        4
uncertain significance, likely pathogenic, pathogenic                 3
not provided                                                          3
likely pathogenic, pathogenic                                         2
benign, likely benign, uncertain significance, likely pathogenic      2
uncertain significance, pathogenic, not p

In [10]:
#going to condense the classifcation labels into 4 groups - VUS, Disease Causing, Benign, and NA
#most labels do not have a mix of benign and pathogenic, so we can group them better. 

#breakdown of grouping
#VUS - anything with uncertain in the label (will run this first)
#Disease Causing - pathogenic| likely pathogenic, pathogenic| likely pathogenic, pathogenic, low penetrance
#Benign - |benign, likely benign | benign | likely benign |
#NA - Not Provided (run last) 

#creating a duplicate of Clinvar_classification to make these changes to, so we can keep the original column
BRCA1_SS["clinvar_CL"] = BRCA1_SS["clinvar_classification"]
BRCA1_SS[["clinvar_CL","clinvar_classification"]].head(5)

C:\Users\knigh\AppData\Local\Temp\ipykernel_183348\3829767905.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  BRCA1_SS["clinvar_CL"] = BRCA1_SS["clinvar_classification"]


,clinvar_CL,clinvar_classification
0,not provided,not provided
1,uncertain significance,uncertain significance
2,"uncertain significance, likely pathogenic, pat...","uncertain significance, likely pathogenic, pat..."
3,"uncertain significance, likely pathogenic, not...","uncertain significance, likely pathogenic, not..."
4,"uncertain significance, pathogenic, not provided","uncertain significance, pathogenic, not provided"


In [11]:
#changing the labels of the clinvar_CL (concordance labels) to match the groups made eariler
#starting with the uncertain group
BRCA1_SS.loc[BRCA1_SS["clinvar_CL"].str.contains("uncertain", na=False), "clinvar_CL"] = "VUS"

#checking the counts, should be 336
BRCA1_SS["clinvar_CL"].value_counts()

clinvar_CL
VUS                                              336
benign, likely benign                             46
benign                                             5
pathogenic                                         4
not provided                                       3
likely pathogenic, pathogenic                      2
likely pathogenic, pathogenic, low penetrance      1
likely benign                                      1
Name: count, dtype: int64

In [12]:
#now changing the pathogenic rows to be disease causing
BRCA1_SS.loc[BRCA1_SS["clinvar_CL"].str.contains("pathogenic", na=False), "clinvar_CL"] = "Disease Causing"

#checking count, there should be 7
BRCA1_SS["clinvar_CL"].value_counts()

clinvar_CL
VUS                      336
benign, likely benign     46
Disease Causing            7
benign                     5
not provided               3
likely benign              1
Name: count, dtype: int64

In [13]:
#now changing the benign group
BRCA1_SS.loc[BRCA1_SS["clinvar_CL"].str.contains("benign",na=False),"clinvar_CL"] = 'Benign'

#checking count, should be 52
BRCA1_SS["clinvar_CL"].value_counts()

clinvar_CL
VUS                336
Benign              52
Disease Causing      7
not provided         3
Name: count, dtype: int64

In [14]:
#Lastly changing the Not Provide to an NA
BRCA1_SS.loc[BRCA1_SS["clinvar_CL"].str.contains("not provided",na=False),"clinvar_CL"] = pd.NA

#counting NAs, should be 31
print(BRCA1_SS["clinvar_CL"].isna().sum())

#checking full count
BRCA1_SS["clinvar_CL"].value_counts()

31


clinvar_CL
VUS                336
Benign              52
Disease Causing      7
Name: count, dtype: int64

In [15]:
#Now updating the REVEL column
#REVEL has a score scale for their classifications:
#Range	Classification:
    #≥ 0.644	Pathogenic supporting
    #0.643 - 0.291	Neutral
    #≤ 0.290	Benign supporting

#Copying the Revel Column to modify
BRCA1_SS['revel_CL'] = BRCA1_SS['revel']
BRCA1_SS[['revel_CL',"revel"]].head(5)

C:\Users\knigh\AppData\Local\Temp\ipykernel_183348\738839862.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  BRCA1_SS['revel_CL'] = BRCA1_SS['revel']


,revel_CL,revel
0,0.562,0.562
1,0.690,0.690
2,0.862,0.862
3,0.789,0.789
4,0.812,0.812


In [16]:
#Updating the revel_CL column with the concordance labels
#Revel score break down.
#≥ 0.644	Pathogenic supporting
#0.643 - 0.291	Neutral (it is benign leaning, so for concordance purposes we will label as benign)
#≤ 0.290	Benign supporting

#assigning the revel scores to their labels based on the score scale
BRCA1_SS.loc[BRCA1_SS['revel'] >= .644, 'revel_CL'] = 'Disease Causing'
BRCA1_SS.loc[BRCA1_SS['revel'] <= .643, 'revel_CL'] = 'Benign'

print((BRCA1_SS['revel_CL'] == 'Disease Causing').sum())
print((BRCA1_SS['revel_CL'] == 'Benign').sum())

55
371


C:\Users\knigh\AppData\Local\Temp\ipykernel_183348\1222067492.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Disease Causing' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  BRCA1_SS.loc[BRCA1_SS['revel'] >= .644, 'revel_CL'] = 'Disease Causing'


In [ ]:
#Moving on to the Polyphen columns to update them to CLs
#Poly-phen has 3 catagories - Benign, possibly damaging, and probably damanging. We will spilt into two groups Benign and Disease Causing

#copying the two Poly-phen classification columns to modify starting with S1 (student 1)
BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"] = BRCA1_SS["(HumDiv) Polyphen Significance(S1)"]
BRCA1_SS[["(HumDiv) Polyphen Significance(S1)_CL", "(HumDiv) Polyphen Significance(S1)"]].head(5)



C:\Users\knigh\AppData\Local\Temp\ipykernel_183348\2072781891.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"] = BRCA1_SS["(HumDiv) Polyphen Significance(S1)"]


,(HumDiv) Polyphen Significance(S1)_CL,(HumDiv) Polyphen Significance(S1)
0,Possibly Damaging,Possibly Damaging
1,Benign,Benign
2,Probably Damaging,Probably Damaging
3,Possibly Damaging,Possibly Damaging
4,Probably Damaging,Probably Damaging


In [ ]:
#now S2 (student 2)
BRCA1_SS["(HumDiv) Polyphen Significance(S2)_CL"] = BRCA1_SS["(HumDiv) Polyphen Significance(S2)"]
BRCA1_SS[["(HumDiv) Polyphen Significance(S2)_CL", "(HumDiv) Polyphen Significance(S2)"]].head(5)



C:\Users\knigh\AppData\Local\Temp\ipykernel_183348\959369395.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  BRCA1_SS["(HumDiv) Polyphen Significance(S2)_CL"] = BRCA1_SS["(HumDiv) Polyphen Significance(S2)"]


,(HumDiv) Polyphen Significance(S2)_CL,(HumDiv) Polyphen Significance(S2)
0,Possibly Damaging,Possibly Damaging
1,Benign,Benign
2,Probably Damaging,Probably Damaging
3,Possibly Damaging,Possibly Damaging
4,Probably Damaging,Probably Damaging


In [19]:

#updating the Polyphen classifications to CLs

#starting with (HumDiv) Polyphen Significance(S1)
#We already have a benign classification, so we only need to updating the damaging classes to disease causing
BRCA1_SS.loc[BRCA1_SS["(HumDiv) Polyphen Significance(S1)"].str.contains("Damaging",na=False),"(HumDiv) Polyphen Significance(S1)_CL"] = 'Disease Causing'

#Next (HumDiv) Polyphen Significance(S2)
BRCA1_SS.loc[BRCA1_SS["(HumDiv) Polyphen Significance(S2)"].str.contains("Damaging",na=False),"(HumDiv) Polyphen Significance(S2)_CL"] = 'Disease Causing'
BRCA1_SS.loc[BRCA1_SS["(HumDiv) Polyphen Significance(S2)"].str.contains("benign",na=False),"(HumDiv) Polyphen Significance(S2)_CL"] = 'Benign'


#comparing counts
BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"].value_counts()

(HumDiv) Polyphen Significance(S1)_CL
Benign             217
Disease Causing    209
Name: count, dtype: int64

In [20]:
BRCA1_SS["(HumDiv) Polyphen Significance(S2)_CL"].value_counts()

(HumDiv) Polyphen Significance(S2)_CL
Benign             217
Disease Causing    209
Name: count, dtype: int64

### All columns are now updated

## Comparing Corcordance Labels between Ploy-phen students, In-silco tools, and Cinvar

## Comparing Poly-phen student scores first

In [ ]:
#taking student 1 and 2 poly-phen columns that we updated eariler with concordance labels and comparing them against eachother
(BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"] == BRCA1_SS["(HumDiv) Polyphen Significance(S2)_CL"]).sum()

#student Poly-phen labels are in concordance

np.int64(426)

In [22]:
#also comparing their scores
print((BRCA1_SS["Score(S1)"] == BRCA1_SS["Score(S2)"]).sum())
print((BRCA1_SS["Sensitivity(S1)"] == BRCA1_SS["Sensitivity(S2)"]).sum())
print((BRCA1_SS["FPR Score(S1)"] == BRCA1_SS["FPR Score(S2)"]).sum())
print((BRCA1_SS["Specificity(S1)"] == BRCA1_SS["Specificity(S2)"]).sum())

423
424
425
424


In [23]:
#there are some scores that don't align, let's look at them
BRCA1_SS.loc[BRCA1_SS["Score(S1)"] != BRCA1_SS["Score(S2)"],["Score(S1)","Score(S2)"]]

#these look mostly like typing errors

,Score(S1),Score(S2)
71,0.255,0.2250
141,0.883,0.8330
324,0.769,0.0769


In [24]:
BRCA1_SS.loc[BRCA1_SS["Sensitivity(S1)"] != BRCA1_SS["Sensitivity(S2)"],["Sensitivity(S1)","Sensitivity(S2)"]]
#these also look mostly like typing errors

,Sensitivity(S1),Sensitivity(S2)
43,0.11800,0.18000
98,0.00574,0.00547


In [25]:
BRCA1_SS.loc[BRCA1_SS["FPR Score(S1)"] != BRCA1_SS["FPR Score(S2)"],["FPR Score(S1)","FPR Score(S2)"]]
#these also look mostly like typing errors

,FPR Score(S1),FPR Score(S2)
52,0.922,0.992


In [26]:
BRCA1_SS.loc[BRCA1_SS["Specificity(S1)"] != BRCA1_SS["Specificity(S2)"],["Specificity(S1)","Specificity(S2)"]]
#these look like a rounding error and possibly an actual discordance
#looking at the actual row (#178) I can see it is a typing error. A score from another variant was copied there. All other variables align

,Specificity(S1),Specificity(S2)
52,0.078,0.008
178,0.237,0.198


### Comparing Polyphen to Clinvar

In [ ]:
#since the labels for Polyphen were in total agreement, I will just use the S1 column for comparision
print((BRCA1_SS["clinvar_CL"] == BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"]).sum())

#looking at the discordance
BRCA1_SS.loc[(BRCA1_SS["clinvar_CL"].isin(["Benign", "Disease Causing"])) & #this line is here to skip over the VUS classifications
    (BRCA1_SS["clinvar_CL"] != BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"]),["clinvar_CL", "(HumDiv) Polyphen Significance(S1)_CL"]]

#Clinvar and Poly-phen align for 40 values. That is 67.797% of clinvar that is not VUS or missing

40


,clinvar_CL,(HumDiv) Polyphen Significance(S1)_CL
11,Benign,Disease Causing
19,Benign,Disease Causing
29,Benign,Disease Causing
49,Benign,Disease Causing
50,Benign,Disease Causing
55,Benign,Disease Causing
133,Benign,Disease Causing
164,Benign,Disease Causing
178,Benign,Disease Causing
179,Benign,Disease Causing


### Comparing Revel to Polyphen 

In [ ]:
#since the labels for Polyphen were in total agreement, I will just use the S1 column for comparision again for revel
print((BRCA1_SS["revel_CL"] == BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"]).sum())

#looking at the discordance
BRCA1_SS.loc[(BRCA1_SS["revel_CL"] != BRCA1_SS["(HumDiv) Polyphen Significance(S1)_CL"]),
             ["revel_CL", "(HumDiv) Polyphen Significance(S1)_CL"]]

#260 concordance, 260/426 = 61%, 166 discordant

260


,revel_CL,(HumDiv) Polyphen Significance(S1)_CL
0,Benign,Disease Causing
1,Disease Causing,Benign
5,Benign,Disease Causing
6,Benign,Disease Causing
8,Benign,Disease Causing
...,...,...
410,Benign,Disease Causing
411,Benign,Disease Causing
414,Disease Causing,Benign
417,Benign,Disease Causing


### Comparing Revel to Clinvar

In [ ]:
#looking at the rows that clinvar and revel have the same labels of disease causing or benign
print((BRCA1_SS["clinvar_CL"] == BRCA1_SS["revel_CL"]).sum())

#looking at the discordance
BRCA1_SS.loc[(BRCA1_SS["clinvar_CL"].isin(["Benign", "Disease Causing"])) & #this line is here to skip over the VUS classifications
    (BRCA1_SS["clinvar_CL"] != BRCA1_SS["revel_CL"]),["clinvar_CL", "revel_CL"]]

#Clinvar and revel align for 52 values out of 59. That is 88.1356% of clinvar that is not VUS or missing

52


,clinvar_CL,revel_CL
11,Benign,Disease Causing
31,Benign,Disease Causing
50,Benign,Disease Causing
232,Benign,Disease Causing
349,Benign,Disease Causing
366,Benign,Disease Causing
367,Benign,Disease Causing


### Adding the created labels back onto the original df for exporting to model with in another script

In [ ]:
#Keeping only the key col and the newly added lablel columns from the subset
cols_to_move = ['vid', 'clinvar_CL', 'revel_CL', 
               '(HumDiv) Polyphen Significance(S1)_CL', 
               '(HumDiv) Polyphen Significance(S2)_CL']

#separating the label columns from BRCA1_SS df
subset = BRCA1_SS[cols_to_move]

#Merge with original dataframe on 'vid'
df_final = BRCA1_UG_AN.merge(subset, on='vid', how='left')

#looking at df
df_final.head()

#Data Preview removed to comply with All of Us data user policies.

In [33]:
#exporting the data
df_final.to_csv("C:/Users/knigh/Documents/UHD MSDA/Capstone/Concordance/2026_BRCA1_ConcordanceLabels.csv")